In [1]:
import warnings
warnings.simplefilter(action='ignore')

import argparse
import os,cv2
import random,numpy,pandas
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [2]:
import argparse
import os
import random,numpy
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
import torchvision.utils as vutils
from torchvision.models.feature_extraction import create_feature_extractor
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [3]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

Random Seed:  999


In [4]:
ngpu=1
ngf,nc = 3,3
ndf = 64

transform = transforms.Compose([
    transforms.Resize((450,450)),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [5]:
device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

In [6]:
class EffnetModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        effnet = torchvision.models.efficientnet_v2_m(weights=torchvision.models.efficientnet.EfficientNet_V2_M_Weights.DEFAULT)
        self.model = create_feature_extractor(effnet, ['flatten'])
        self.nn_fracture = torch.nn.Sequential(
            torch.nn.Linear(1280, 1)
        )
    def forward(self, x):
        x = self.model(x)['flatten']
        x = self.nn_fracture(x)
        return x

In [7]:
EFF_NET = EffnetModel().float()
EFF_NET= nn.DataParallel(EFF_NET).to(device)

Downloading: "https://download.pytorch.org/models/efficientnet_v2_m-dc08266a.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_m-dc08266a.pth
100%|██████████| 208M/208M [00:01<00:00, 197MB/s]


In [8]:
fsc_submission=pandas.read_csv("/kaggle/input/ai-vs-human-generated-dataset/test.csv")['id']

In [9]:
sub = ([0]*len(fsc_submission))
file_ = ['9.83672944130376e-05 1.pth']

EFF_NET.load_state_dict(torch.load(f"/kaggle/input/detect-ai-vs-human-generated-images-mode-2/{file_[0]}",
                                   weights_only=False,
                                   map_location=torch.device('cpu')))

for i,j in zip(fsc_submission,range(len(fsc_submission))):
    #print(i.split("/")[1])
    img = transform(Image.open(f'/kaggle/input/detect-ai-vs-human-generated-images-model-1/test/{i.split("/")[1]}')).reshape((1, 3, 450, 450)).float().to(device)    
    sub[j] += EFF_NET(img).sigmoid().cpu().detach().numpy()[0][0]

In [10]:
submission=pandas.DataFrame({'id' : fsc_submission, 
                             'label' : numpy.array(sub)})

for i in range(len(submission['label'])):
    if submission.iloc[[i]]['label'].to_numpy()[0]>0.5:
        submission.loc[i,'label']=0
    else:
        submission.loc[i,'label']=1
        
pandas.DataFrame(submission).to_csv(f"submission.csv", index=False)
pandas.DataFrame(submission)

,id,label
0,test_data_v2/1a2d9fd3e21b4266aea1f66b30aed157.jpg,0.0
1,test_data_v2/ab5df8f441fe4fbf9dc9c6baae699dc7.jpg,0.0
2,test_data_v2/eb364dd2dfe34feda0e52466b7ce7956.jpg,0.0
3,test_data_v2/f76c2580e9644d85a741a42c6f6b39c0.jpg,0.0
4,test_data_v2/a16495c578b7494683805484ca27cf9f.jpg,0.0
...,...,...
5535,test_data_v2/483412064ff74d9d9472d606b65976d9.jpg,0.0
5536,test_data_v2/c0b49ba4081a4197b422dac7c15aea7f.jpg,0.0
5537,test_data_v2/01454aaedec140c0a3ca1f48028c41cf.jpg,0.0
5538,test_data_v2/e9adfea8b67e4791968c4c2bdd8ec343.jpg,1.0
